# Build a Looped MoE: quickstart

A **looped transformer** runs the same middle layers several times. Inkling 2 Mini, an open-weights looped
mixture-of-experts model, also drops the single residual stream: layers are grouped into blocks, every sublayer reads
its own learned mix of the finished blocks, every pass of the looped core restarts from the same input, and the
previous pass reaches the next only through a gated message.

This notebook is a short tour of the course at https://github.com/albertlieyingadrian/build-a-looped-moe, which
rebuilds those ideas at toy scale (an Inkling-style model; not affiliated with Thinking Machines):

1. clone the course and run the tests for stages 01-11;
2. look at three of the ideas from the command line: blocks, the restart-from-the-prelude loop, the closed gate;
3. train the looped MoE and a plain loop briefly, and read every layer visit with three lenses.

Everything runs on a CPU.

In [ ]:
# Setup: clone the course (or use this checkout) and install what Colab doesn't already have.
import os
import sys

if os.path.exists("../course-definition.yml"):
    %cd ..
elif not os.path.exists("course-definition.yml"):
    if not os.path.isdir("build-a-looped-moe"):
        !git clone -q --depth 1 https://github.com/albertlieyingadrian/build-a-looped-moe
    %cd build-a-looped-moe
try:
    import torch
except ImportError:
    !{sys.executable} -m pip install -q torch
!{sys.executable} -m pip install -q -r requirements.txt
import torch

print("torch", torch.__version__)

## 1. The first eleven stages' tests

Each stage adds one piece and has its own tests. `solutions/python/NN-<slug>/code/` is the reference program right
after stage NN. The tests run a program through its command line, the way an automated grader would.

| stages | what you build |
|---|---|
| 01 | recap: a plain looped GPT with one residual stream, and the chain puzzle |
| 02-05 | blocks and the active block, depth attention, separate q/k/v mixes, the head's `final_input` mix |
| 06-08 | a sigmoid top-k router, a mixture of experts, load balancing through the router bias |
| 09-11 | the loop (every pass restarts from the prelude), the message bank, the gated message |
| 12-13 | train against a plain loop; lenses when there is no single residual stream |
| 14 | the Inkling 2 Mini layout (48 visits) and a logit-for-logit check against the one-file model |

In [ ]:
!make test-upto STAGE=11 PY={sys.executable}

## 2. Three ideas from the command line

**Blocks.** Sublayer outputs add up only inside the block being built; at each boundary the block is frozen into a
list. Add the list back up and you get the ordinary residual stream, exactly.

In [ ]:
P = "solutions/python/14-layout/code/your_program.sh"
os.environ["PYTHON"] = sys.executable
!./{P} blocks

**The loop restarts.** Every pass of the core starts again from the prelude's blocks. With messages switched off,
running the core 4 times gives exactly the same output as running it once. Inkling 2 Mini's layout, 24 layers with
layers 6-17 run three times, gives 48 layer visits.

In [ ]:
!./{P} loop

**The gate starts closed.** Messages from the previous pass enter through `x <- (1 - g_forget) x + g_message m`,
with both gates zero at initialisation, so a new model behaves like a non-looped one until training opens them.

In [ ]:
!./{P} gate

## 3. Train briefly, then read every visit

`compare` trains four models on the chain puzzle at matched active parameters. Here only the plain loop and the
looped MoE, for 300 steps: about a minute or two on a CPU. That is far too short to get past the "guess one of the two
digits" plateau (about 0.55); the point is to see the lens machinery. Stage 12 has the numbers from a longer run.

In [ ]:
import time

t = time.time()
!./{P} compare --only plainloop,loopmoe --steps 300 --k-max 3 --out-dir runs/colab
!./{P} lens --n 300 --out-dir runs/colab
print(f"took {time.time() - t:.0f} s")

Three readings per visit for the looped MoE (the active block, the visit's own MoE-input mix, and the head's
`final_input` mix applied early), one for the plain loop (its residual stream). The last point, `out`, is the model's
real output.

In [ ]:
import json

import matplotlib.pyplot as plt

res = json.load(open("runs/colab/lens.json"))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True, gridspec_kw={"width_ratios": [3, 2]})
for k, label in (("act_in", "logit lens (active block)"), ("mlp_in", "visit's MoE-input mix"), ("mix", "mix lens")):
    axes[0].plot(res["loopmoe"][k], "-o", ms=3, label=label)
axes[0].set_xticks(range(len(res["loopmoe"]["visits"])), res["loopmoe"]["visits"], rotation=70, fontsize=7)
axes[0].set(title="looped MoE with block mixing", ylabel="top-1 is the answer at '?'", ylim=(0, 1.02))
axes[1].plot(res["plainloop"]["act_in"], "-o", ms=3, color="C3", label="logit lens (residual stream)")
axes[1].set_xticks(range(len(res["plainloop"]["visits"])), res["plainloop"]["visits"], rotation=70, fontsize=7)
axes[1].set(title="plain loop, one stream")
for ax in axes:
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Where to go next

- Build it yourself: copy `starter_templates/python/code`, open `app/main.py`, and follow the stage descriptions in
  [`stage_descriptions/`](https://github.com/albertlieyingadrian/build-a-looped-moe/tree/main/stage_descriptions),
  starting with [stage 01](https://github.com/albertlieyingadrian/build-a-looped-moe/blob/main/stage_descriptions/base-01-recap.md).
  Check each stage with `!make test STAGE=NN DIR=your/folder PY={sys.executable}`.
- The same model in one file: https://github.com/albertlieyingadrian/nanoloop-moe.
- The course before this one: https://github.com/albertlieyingadrian/build-a-looped-transformer.